In [1]:
import torch
import pandas as pd

import sys
sys.path.append("/Users/gabriel/Documents/Apps/PolyGraphPy/")
from polygraphpy.gnn.pre_processing import PreProcess

# Setup
preprocess = PreProcess(
    input_csv='polarizability_data_monomer.csv',
    train_input_data_path='../polygraphpy/data/training_input_data/',
    polymer_type='monomer',
    target='static_polarizability',
    gnn_output_path='./'
)

df = preprocess.run()
atoms_list, bonds_list = preprocess.extract_atoms_and_bonds_features_from_monomer_smiles()
atom_encoder = preprocess.make_encoder(pd.DataFrame(atoms_list).drop_duplicates().reset_index(drop=True))
bond_encoder = preprocess.make_encoder(pd.DataFrame(bonds_list).drop_duplicates().reset_index(drop=True))

model = torch.load('../polygraphpy/data/gnn_output/model_gcn.pt', weights_only=False)
print(model)

Reading GNN input file.
Removing outliers...
Making data standardization...
Extracting unique features from atoms and bonds.


100%|██████████| 9003/9003 [00:02<00:00, 3415.58it/s]


Making feature encoder.
Making feature encoder.
Training data preparation starting. 9003 to go.


9003it [00:27, 328.76it/s]


Training data preparation finished.
Extracting unique features from atoms and bonds.


100%|██████████| 9003/9003 [00:02<00:00, 3674.10it/s]


Making feature encoder.
Making feature encoder.
GCN(
  (conv1): GCNConv(75, 225)
  (conv2): GCNConv(225, 225)
  (conv3): GCNConv(225, 225)
  (lin1): Linear(in_features=225, out_features=225, bias=True)
  (lin2): Linear(in_features=225, out_features=225, bias=True)
  (lin3): Linear(in_features=225, out_features=225, bias=True)
  (output): Linear(in_features=225, out_features=1, bias=True)
)


In [4]:
from rdkit import Chem
from rdkit.Chem import BRICS, Descriptors
import random
import time
from torch_geometric.data import Batch
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
import torch
from torch_geometric.data import Data
import numpy as np
from sklearn.preprocessing import MinMaxScaler

class FragmentGA:
    def __init__(self, csv_path, model, preprocess, atom_encoder, bond_encoder, population_size=30, target_polarizability=0.43):
        self.df = pd.read_csv(csv_path)
        self.target_value = target_polarizability
        self._pre_process()

        self.model = model.eval()
        self.preprocess = preprocess
        self.atom_encoder = atom_encoder
        self.bond_encoder = bond_encoder
        self.population_size = population_size
        self.fragments = self._extract_fragments()
        print("Fragments sample: ")
        print(self.fragments[:25])
        self.device = next(model.parameters()).device

    def _pre_process(self,):
        scaler = MinMaxScaler()
        self.df['target_scaled'] = scaler.fit_transform(self.df['static_polarizability'].values.reshape(-1,1))
        atoms_number = []

        for i in self.df['smiles'].values:
            mol = Chem.MolFromSmiles(i)
            mol_with_hs = Chem.AddHs(mol)
            num_all_atoms = mol_with_hs.GetNumAtoms()

            atoms_number.append(num_all_atoms)
        
        self.df['number_of_atoms'] = atoms_number

        self.df = self.df[self.target_value <= self.df['target_scaled']*1.20].reset_index(drop=True)
        self.df = self.df[self.target_value >= self.df['target_scaled']*0.80].reset_index(drop=True)
        self.df = self.df[self.df['number_of_atoms'] <= 40].reset_index(drop=True)

    def _extract_fragments(self):
        start_time = time.time()
        all_frags = set()
        acrylate_core = Chem.MolFromSmarts('C=C-C(=O)O-[*]')
        for smi in tqdm(self.df['smiles']):
            mol = Chem.MolFromSmiles(smi, sanitize=True)
            if mol is None:
                continue
            try:
                Chem.RemoveStereochemistry(mol)
                if not mol.HasSubstructMatch(acrylate_core):
                    continue
                frags = BRICS.BRICSDecompose(mol, minFragmentSize=3, keepNonLeafNodes=True)
                for f in frags:
                    frag_mol = Chem.MolFromSmiles(f, sanitize=True)
                    if frag_mol and '*' in f and Descriptors.MolWt(frag_mol) < 200:  # Filter by molecular weight
                        all_frags.add(f)
            except:
                continue
        fragments = list(all_frags)[:800]
        print(f"Extracted {len(fragments)} valid R-group fragments in {time.time() - start_time:.2f} seconds")
        return fragments
    
    def _mol_to_data(self, smiles):
        try:
            atoms = []
            bonds = []
            m1 = Chem.MolFromSmiles(smiles, sanitize=True)
            if m1 is None:
                print(f"Invalid SMILES: {smiles}")
                return None
            m1 = Chem.AddHs(m1)
            
            atoms = self.preprocess.get_nodes_information(m1, [], chain_size=0)
            if not atoms:
                print(f"No atoms extracted for SMILES: {smiles}")
                return None
            df_nodes = pd.DataFrame(atoms)
            nodes_features = pd.DataFrame(self.atom_encoder.transform(df_nodes.drop(['idx'], axis=1)).toarray())
            zero_vector = np.zeros((nodes_features.shape[0], 1))
            nodes_features = pd.concat([nodes_features, pd.DataFrame(zero_vector)], axis=1)
            nodes_features = pd.concat([nodes_features, pd.DataFrame(zero_vector)], axis=1)
            x = torch.tensor(nodes_features.astype('float32').values)
            
            bonds = self.preprocess.get_bonds_information(m1, [])
            if not bonds:
                print(f"No bonds extracted for SMILES: {smiles}")
                return None
            df_bonds = pd.DataFrame(bonds)
            edge_index = torch.tensor([
                df_bonds.begin_idx.to_list() + df_bonds.end_idx.to_list(),
                df_bonds.end_idx.to_list() + df_bonds.begin_idx.to_list()
            ])
            
            edge_attrs = df_bonds[['type', 'is_conjugated', 'is_aromatic']]
            edge_attrs = pd.concat([edge_attrs, edge_attrs.sort_index(ascending=False)])
            edge_attr = torch.tensor(self.bond_encoder.transform(edge_attrs).toarray(), dtype=torch.float32)
            
            edge_weight = torch.tensor([1.0] * edge_index.shape[1], dtype=torch.float32)
            
            mol_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, edge_weight=edge_weight)
            mol_data.validate()
            return mol_data
        except Exception as e:
            print(f"Error in _mol_to_data for SMILES {smiles}: {str(e)}")
            return None

    def _evaluate_fitness_batch(self, smiles_list, target_polarizability):
        start_time = time.time()
        data_list = []
        valid_smiles = []
        for smi in smiles_list:
            data = self._mol_to_data(smi)
            if data is not None:
                data_list.append(data)
                valid_smiles.append(smi)
        if not data_list:
            return [(smi, -1.0) for smi in smiles_list]
        batch = Batch.from_data_list(data_list).to(self.device)
        with torch.no_grad():
            predictions = self.model(batch.x, batch.edge_index, batch.edge_weight, batch.batch).cpu().numpy()
        scores = [-abs(pred - target_polarizability) if pred > 0 else -1.0 for pred in predictions]
        #print(f"Fitness evaluation took {time.time() - start_time:.2f} seconds")
        return list(zip(valid_smiles, scores)) + [(smi, -1.0) for smi in smiles_list if smi not in valid_smiles]

# Standalone function for parallelization
def build_molecule(fragments):
    from rdkit import Chem
    from rdkit.Chem import BRICS
    acrylate_core = Chem.MolFromSmiles('C=CC(=O)O*')
    for attempt in range(100):
        r_frags = random.sample(fragments, k=random.randint(1, 3))[:100]
        try:
            mol_frags = [Chem.MolFromSmiles(f, sanitize=True) for f in r_frags]
            mol_frags.append(acrylate_core)
            if None in mol_frags:
                continue
            new_mol = BRICS.BRICSBuild(mol_frags)
            for mol in new_mol:
                Chem.RemoveStereochemistry(mol)
                smi = Chem.MolToSmiles(mol, isomericSmiles=False)
                mol = Chem.MolFromSmiles(smi, sanitize=True)
                if mol and mol.HasSubstructMatch(Chem.MolFromSmarts('C=C-C(=O)O')):
                    Chem.SanitizeMol(mol)
                    return smi
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with fragments {r_frags}: {str(e)}")
            continue
    return None

# Modified run with joblib parallelization
def run_parallel(self, generations=10, target_polarizability=0.555):
    start_time = time.time()
    print("Generating initial population...")
    population = Parallel(n_jobs=-1, backend='loky')(delayed(build_molecule)(self.fragments) for _ in tqdm(range(self.population_size)))
    population = [p for p in population if p is not None]
    print(f"Initial population generated in {time.time() - start_time:.2f} seconds")
    if not population:
        print("Initial population empty. Check fragment generation.")
        return []

    for gen in tqdm(range(generations)):
        gen_start = time.time()
        #print(f"Generation {gen + 1}")
        fitness_scores = self._evaluate_fitness_batch(population, target_polarizability)
        fitness_scores.sort(key=lambda x: x[1], reverse=True)
        top_individuals = fitness_scores[:self.population_size // 2]
        if len(top_individuals) == 0:
            print("No valid molecules in this generation. Reinitializing population...")
            population = Parallel(n_jobs=-1, backend='loky')(delayed(build_molecule)(self.fragments) for _ in range(self.population_size))
            population = [p for p in population if p is not None]
            continue

        top_frags = set()
        for smi, _ in top_individuals:
            mol = Chem.MolFromSmiles(smi, sanitize=True)
            if mol is None:
                continue
            try:
                top_frags.update(BRICS.BRICSDecompose(mol, minFragmentSize=3))
            except:
                continue
        original_fragments = self.fragments
        self.fragments = list(top_frags)[:500] if top_frags else original_fragments
        #print(f"Crossover fragments: {len(self.fragments)}")
        new_population = Parallel(n_jobs=-1, backend='loky')(delayed(build_molecule)(self.fragments) for _ in range(self.population_size))
        self.fragments = original_fragments
        population = [p for p in new_population if p is not None]
        if not population:
            print("Crossover failed to produce valid molecules. Reinitializing population...")
            population = Parallel(n_jobs=-1, backend='loky')(delayed(build_molecule)(self.fragments) for _ in range(self.population_size))
            population = [p for p in population if p is not None]
        #print(f"Generation {gen + 1} completed in {time.time() - gen_start:.2f} seconds")

    print(f"Total runtime: {time.time() - start_time:.2f} seconds")
    return fitness_scores

FragmentGA.run_parallel = run_parallel

In [32]:
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem

target_value = 0.33333
ga = FragmentGA(csv_path='polarizability_data_monomer.csv',
                model=model,
                preprocess=preprocess,
                atom_encoder=atom_encoder,
                bond_encoder=bond_encoder,
                population_size=100,
                target_polarizability=target_value)

#top_molecules = ga.run_parallel(generations=50, target_polarizability=target_value)

for i in top_molecules[:3]:
    print(f"Acrylate SMILES: {i[0]} | Fitness: {i[1][0]:.4f}")
    mol = Chem.MolFromSmiles(i[0])
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) # Generate 3D coordinates
    view = py3Dmol.view(width=400, height=400)
    view.addModel(Chem.MolToMolBlock(mol), 'mol')
    view.setStyle({'stick': {}, 'sphere':{"radius": 0.5, "colorscheme": "Jmol"}}) # Adjust radius as needed
    view.zoomTo()
    view.show()

100%|██████████| 14/14 [00:00<00:00, 298.08it/s]

Extracted 50 valid R-group fragments in 0.05 seconds
Fragments sample: 
['[5*]NC(C)=O', '[1*]C(=O)C(C#N)=CN[5*]', '[7*]Cc1ccc(C=Cc2csc([14*])n2)cc1', '[1*]C(=O)C=Cc1ccccc1Cl', '[14*]c1ccc(-c2ccncc2)s1', '[1*]C(=O)C([7*])C#N', '[1*]C(=O)c1ccc(Cl)cc1Cl', '[3*]OC(=O)C([7*])C#N', '[7*]Cc1ccc(O)c(O)c1', '[7*]C(C#N)C(=O)OC(C)C', '[1*]C(=O)C(C#N)=Cc1ccc([14*])o1', '[11*]SC(F)F', '[5*]NC=C(NC([6*])=O)C(=O)OC', '[1*]C(=O)C(C#N)=CNc1ccc([16*])cc1', '[5*]Nc1nc(C[7*])cs1', '[7*]Cc1onc(C)c1[N+](=O)[O-]', '[1*]C(=O)C([7*])C=Nc1nc(N)c(F)cc1F', '[16*]c1ccc(SC(F)F)cc1', '[3*]OC(=O)C[7*]', '[14*]c1ccc([14*])s1', '[7*]Cc1ccc(Cl)cc1', '[1*]C(=O)C[7*]', '[1*]C(=O)C=Cc1ccc(C[7*])cc1', '[7*]C(O)c1cc(F)c(F)c(Cl)c1F', '[5*]NC=C(N[5*])C(=O)OC']
Acrylate SMILES: Nc1nc(N=CC(C(=O)OC(=O)c2ccc3c(c2)oc(=O)c2ccccc23)=C(C=Nc2nc(N)c(F)cc2F)C(=O)Oc2ccc3c(c2)oc(=O)c2ccccc23)c(F)cc1F | Fitness: -0.0019


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Acrylate SMILES: N#CC(=CNc1ccc2c(c1)oc(=O)c1ccccc12)C(=O)Oc1ccc2c(c1)oc(=O)c1ccccc12 | Fitness: -0.0034


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Acrylate SMILES: COC(=O)C(=CNC(=O)c1ccc(Cl)cc1Cl)NC(=O)c1ccc(Cl)cc1Cl | Fitness: -0.0038


3Dmol.js failed to load for some reason. Please check your browser console for error messages.